# Mini aplicacao de RAG com Hugging Face

**Tema:** Retrieval-Augmented Generation (RAG)

Este notebook demonstra uma cadeia simples de RAG usando bibliotecas do ecossistema Hugging Face em Python.

## 1. Instalar dependencias

No Google Colab, descomente a linha abaixo se necessario.

In [15]:
# !pip install transformers sentence-transformers torch scikit-learn

## 2. Importar bibliotecas

In [16]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import numpy as np

## 3. Criar uma pequena base de documentos

Para simplificar a atividade, vamos usar poucos textos curtos.

In [17]:
documents = [
    "RAG significa Retrieval-Augmented Generation. A tecnica combina recuperacao de informacao com geracao de texto.",
    "Embeddings sao representacoes numericas de textos. Eles permitem comparar similaridade semantica entre frases e documentos.",
    "O modelo all-MiniLM-L6-v2 e bastante usado para gerar embeddings leves e eficientes.",
    "O modelo FLAN-T5 e uma familia de modelos instruidos capazes de responder perguntas e seguir comandos textuais.",
    "Em um pipeline de RAG, primeiro recuperamos contexto relevante e depois passamos esse contexto ao modelo gerador.",
    "RAG pode ajudar a reduzir alucinacoes porque a resposta passa a usar textos recuperados como apoio."
]

for i, doc in enumerate(documents, start=1):
    print(f"Documento {i}: {doc}")

Documento 1: RAG significa Retrieval-Augmented Generation. A tecnica combina recuperacao de informacao com geracao de texto.
Documento 2: Embeddings sao representacoes numericas de textos. Eles permitem comparar similaridade semantica entre frases e documentos.
Documento 3: O modelo all-MiniLM-L6-v2 e bastante usado para gerar embeddings leves e eficientes.
Documento 4: O modelo FLAN-T5 e uma familia de modelos instruidos capazes de responder perguntas e seguir comandos textuais.
Documento 5: Em um pipeline de RAG, primeiro recuperamos contexto relevante e depois passamos esse contexto ao modelo gerador.
Documento 6: RAG pode ajudar a reduzir alucinacoes porque a resposta passa a usar textos recuperados como apoio.


## 4. Gerar embeddings dos documentos

In [18]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
document_embeddings = embedding_model.encode(documents)

print("Formato da matriz de embeddings:", np.array(document_embeddings).shape)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4674.56it/s]


Formato da matriz de embeddings: (6, 384)


## 5. Definir a funcao de recuperacao

A funcao abaixo recebe uma pergunta, calcula o embedding da pergunta e retorna os documentos mais parecidos.

In [19]:
def retrieve_relevant_docs(query, top_k=2):
    query_embedding = embedding_model.encode([query])
    scores = cosine_similarity(query_embedding, document_embeddings)[0]
    top_indices = scores.argsort()[::-1][:top_k]

    results = []
    for idx in top_indices:
        results.append({
            "document": documents[idx],
            "score": float(scores[idx])
        })
    return results

## 6. Carregar o modelo gerador

In [20]:
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
generator_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 3427.64it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


## 7. Montar a funcao completa de RAG

In [21]:
def answer_with_rag(question, top_k=2):
    retrieved = retrieve_relevant_docs(question, top_k=top_k)
    context = "\n".join([item["document"] for item in retrieved])

    prompt = f"""
Use apenas o contexto abaixo para responder a pergunta.

Contexto:
{context}

Pergunta: {question}
Resposta:
"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    outputs = generator_model.generate(**inputs, max_new_tokens=80)
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return {
        "question": question,
        "context": context,
        "retrieved": retrieved,
        "answer": result
    }

## 8. Testar a aplicacao

In [22]:
question = "Como o RAG ajuda a reduzir alucinacoes em modelos de linguagem?"
response = answer_with_rag(question)

print("Pergunta:")
print(response["question"])
print("\nDocumentos recuperados:")
for item in response["retrieved"]:
    print(f"- score={item['score']:.4f} | {item['document']}")

print("\nContexto enviado ao modelo:")
print(response["context"])

print("\nResposta gerada:")
print(response["answer"])

Pergunta:
Como o RAG ajuda a reduzir alucinacoes em modelos de linguagem?

Documentos recuperados:
- score=0.6504 | RAG pode ajudar a reduzir alucinacoes porque a resposta passa a usar textos recuperados como apoio.
- score=0.5302 | Em um pipeline de RAG, primeiro recuperamos contexto relevante e depois passamos esse contexto ao modelo gerador.

Contexto enviado ao modelo:
RAG pode ajudar a reduzir alucinacoes porque a resposta passa a usar textos recuperados como apoio.
Em um pipeline de RAG, primeiro recuperamos contexto relevante e depois passamos esse contexto ao modelo gerador.

Resposta gerada:
A resposta passa a usar textos recuperados como apoio. Em um pipeline de RAG, primeiro recuperamos contexto relevante e passamos esa contexto ao modelo gerador.


## 9. Conclusao

Este exemplo mostra uma forma simples de construir RAG com Hugging Face:

- `sentence-transformers` para representar documentos em embeddings;
- similaridade cosseno para recuperar contexto;
- `transformers` para gerar a resposta final.

Em um projeto real, a base de documentos poderia vir de PDFs, sites, banco vetorial ou arquivos internos da empresa.